# Phase 1 — Parsing the SemEval-2013 DDI Corpus

**Project:** Drug–Drug Interaction Detection using NLP  

---

## Overview

This notebook parses the **SemEval-2013 Drug–Drug Interaction (DDI) corpus** from raw XML into a clean, structured CSV file ready for NLP preprocessing.

The corpus was created by medical experts at Universidad Carlos III de Madrid. Every sentence is manually annotated with:
- **Drug entity spans** — where each drug name appears in the sentence
- **Interaction pairs** — which two drugs interact
- **Interaction type** — one of five class labels:

| Label | Meaning |
|---|---|
| `mechanism` | How the interaction works biologically |
| `effect` | What clinical effect it causes in the patient |
| `advise` | A recommendation or warning |
| `int` | Interaction exists but type is unspecified |
| `false` | No interaction between this drug pair |

**Pipeline position:**
```
[Phase 1: Parse XML] → Phase 2: EDA → Phase 3: Preprocess → Phase 4: Features → Phase 5: Train Models
```

**Input:** DDICorpus XML files (Train/ and Test/ folders)  
**Output:** `train.csv` and `test.csv` — one row per drug pair per sentence

## 1. Imports

All modules are from Python's standard library, no external packages needed for this phase.
- `os` — file system traversal
- `csv` — writing structured output
- `ElementTree` — XML parsing
- `Counter` — label frequency counting
- `pandas` — data inspection and spot-checking

In [1]:
import os
import csv
from xml.etree import ElementTree as ET
from collections import Counter
import pandas as pd

## 2. Configuration

All paths are defined here. This is the **only section you need to edit** when moving the notebook to a different machine.

- `TRAIN_DIR` / `TEST_DIR` — paths to corpus folders (contain `DrugBank/` and `MedLine/` subfolders)
- `TRAIN_OUTPUT` / `TEST_OUTPUT` — where the parsed CSV files will be saved

In [2]:
TRAIN_DIR    = '../../data/raw/DDICorpus-2013/Train'
TEST_DIR     = '../../data/raw/DDICorpus-2013/Test'
TRAIN_OUTPUT = '../../data/processed/train.csv'
TEST_OUTPUT  = '../../data/processed/test.csv'

for path in [TRAIN_DIR, TEST_DIR]:
    status = 'FOUND' if os.path.exists(path) else 'NOT FOUND — check your path!'
    print(f'{path} => {status}')

../../data/raw/DDICorpus-2013/Train => FOUND
../../data/raw/DDICorpus-2013/Test => FOUND


## 3. XML Structure Overview

Each XML file follows this structure:

```xml
<document id="DrugDDI.d1">
  <sentence id="DrugDDI.d1.s0"
            text="Warfarin may increase bleeding risk when combined with aspirin.">
    <entity id="...e0" charOffset="0-7"   type="drug" text="Warfarin" />
    <entity id="...e1" charOffset="52-58" type="drug" text="aspirin"  />
    <pair   id="...p0" e1="...e0" e2="...e1" ddi="true" type="effect" />
  </sentence>
</document>
```

**Fields extracted per pair:**
- `sentence_text` — the full sentence from the research paper
- `drug1_text` / `drug2_text` — the two drug names in the pair
- `charOffset` — character positions used for offset validation
- `ddi` — whether an interaction exists (`true` / `false`)
- `interaction_type` — the class label used for model training

## 4. Core Parser Functions

### 4.1 Error Handling for Malformed XML

Each file is wrapped in a `try/except` block. If parsing fails, the error is logged to a list and the script continues to the next file. A summary of failed files is printed at the end.

This makes the parser **fault-tolerant** — one bad file no longer destroys a full run.

In [3]:
parse_errors = []


def parse_ddi_file(filepath):
    """
    Parse a single SemEval DDI XML file.
    Returns (rows, offset_warnings) where:
      - rows             : list of dicts, one per drug pair per sentence
      - offset_warnings  : list of annotation mismatches found
    
     wrapped in try/except so one bad file does not crash the run.
     corpus_source column extracted from file path.
     charOffset validated against sentence text.
    """
    rows = []
    offset_warnings = []

    # fault-tolerant XML parsing
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
    except ET.ParseError as e:
        parse_errors.append({'file': filepath, 'error': str(e)})
        print(f'[SKIP] Malformed XML: {os.path.basename(filepath)} — {e}')
        return rows, offset_warnings
    except Exception as e:
        parse_errors.append({'file': filepath, 'error': str(e)})
        print(f'[SKIP] Unexpected error: {os.path.basename(filepath)} — {e}')
        return rows, offset_warnings

    # determine corpus source from file path
    # File path always contains either 'DrugBank' or 'MedLine' as a folder name.
    norm_path = filepath.replace(os.sep, '/')
    corpus_source = 'DrugBank' if 'DrugBank' in norm_path else \
                    'MedLine'  if 'MedLine'  in norm_path else 'Unknown'

    for sentence in root.iter('sentence'):
        sentence_id = sentence.get('id',   '')
        sentence_text = sentence.get('text', '')

        # Collect all drug entities in this sentence
        entities = {}
        for entity in sentence.findall('entity'):
            eid = entity.get('id')
            raw_offset = entity.get('charOffset', '')
            entity_text = entity.get('text', '')

            # validate charOffset against sentence text
            offset_valid = True
            try:
                # charOffset can be "start-end" or "start-end;start-end" (multi-span)
                first_span = raw_offset.split(';')[0]
                start, end = map(int, first_span.split('-'))
                extracted = sentence_text[start:end + 1]
                # Normalise both to lowercase for comparison
                if extracted.lower() != entity_text.lower():
                    offset_valid = False
                    offset_warnings.append({
                        'sentence_id': sentence_id,
                        'entity_id': eid,
                        'expected': entity_text,
                        'extracted': extracted,
                        'char_offset': raw_offset,
                    })
            except Exception:
                offset_valid = False  # Could not parse offset, skip silently

            entities[eid] = {
                'text': entity_text,
                'type': entity.get('type', ''),
                'charOffset': raw_offset,
                'offset_valid': offset_valid,
            }

        # Collect all annotated drug pairs
        for pair in sentence.findall('pair'):
            e1_id = pair.get('e1')
            e2_id = pair.get('e2')
            ddi_flag = pair.get('ddi', 'false')
            ddi_type = pair.get('type', '')

            e1 = entities.get(e1_id, {})
            e2 = entities.get(e2_id, {})

            # Assign interaction label
            if ddi_flag == 'true' and ddi_type:
                label = ddi_type
            elif ddi_flag == 'true':
                label = 'int'
            else:
                label = 'false'

            rows.append({
                'sentence_id': sentence_id,
                'sentence_text': sentence_text,
                'drug1_id': e1_id,
                'drug1_text': e1.get('text', ''),
                'drug1_type': e1.get('type', ''),
                'drug2_id': e2_id,
                'drug2_text': e2.get('text', ''),
                'drug2_type': e2.get('type', ''),
                'ddi': ddi_flag.lower() == 'true',
                'interaction_type': label,
                'corpus_source': corpus_source,     
                'offset_valid': e1.get('offset_valid', True) and e2.get('offset_valid', True), 
                'source_file': os.path.basename(filepath),
            })

    return rows, offset_warnings

### 4.2 Sentence Deduplication Tracking

**Problem:**  
Some sentences appear multiple times in the corpus with different drug pairs. For model training this is intentional and correct. However, knowing the extent of duplication is important — high duplication can cause **data leakage** if the same sentence appears in both train and test sets.

**Fix implemented:**  
`parse_ddi_folder()` now tracks how many times each unique sentence text appears. After parsing, a deduplication report is printed showing:
- Total unique sentences
- How many sentences appear more than once
- The top 5 most repeated sentences

In [4]:
def parse_ddi_folder(data_dir):
    """
    Walk a directory of XML files and parse all of them.
    Handles DrugBank/ and MedLine/ subfolders automatically.

    skips malformed files and logs errors.
    collects and reports offset validation warnings.
    tracks sentence-level duplication across files.
    """
    all_rows = []
    all_warnings = []
    xml_files = []
    sentence_counter = Counter()

    for root_dir, _, files in os.walk(data_dir):
        for fname in files:
            if fname.endswith('.xml'):
                xml_files.append(os.path.join(root_dir, fname))

    print(f'Found {len(xml_files)} XML files in "{data_dir}"')

    for fpath in sorted(xml_files):
        rows, warnings = parse_ddi_file(fpath)
        all_rows.extend(rows)
        all_warnings.extend(warnings)

        # count unique sentence texts from this file
        for r in rows:
            sentence_counter[r['sentence_text'].strip()] += 1

        status = f'{len(rows):4d} pairs'
        print(f'{os.path.basename(fpath):50s} -> {status}')

    # report any files that failed 
    if parse_errors:
        print(f'\n[WARNING] {len(parse_errors)} file(s) failed to parse:')
        for err in parse_errors:
            print(f'{err["file"]} : {err["error"]}')
    else:
        print('\nAll files parsed successfully (no errors).')

    # report offset mismatches 
    if all_warnings:
        print(f'\n[INFO] {len(all_warnings)} charOffset mismatch(es) found in annotations.')
        print('These are annotation inconsistencies in the original corpus.')
        print('Affected rows have offset_valid=False in the output CSV.')
        print('First 3 examples:')
        for w in all_warnings[:3]:
            print(f'Sentence {w["sentence_id"]}: '
                  f'expected "{w["expected"]}" '
                  f'but offset gives "{w["extracted"]}" '
                  f'at {w["char_offset"]}')
    else:
        print('All charOffsets validated successfully.')

    # deduplication report
    duplicated = {s: c for s, c in sentence_counter.items() if c > 1}
    print(f'\n[INFO] Sentence deduplication report:')
    print(f'Total unique sentences : {len(sentence_counter)}')
    print(f'Sentences appearing >1x: {len(duplicated)}')
    if duplicated:
        print('Top 5 most repeated sentences (pair count):')
        for sent, count in sorted(duplicated.items(), key=lambda x: -x[1])[:5]:
            preview = sent[:70] + '...' if len(sent) > 70 else sent
            print(f'({count}x) {preview}')

    return all_rows

### 4.3 — Statistics Printer & CSV Writer

`print_stats()` prints a class distribution bar chart — important for spotting class imbalance early.  

`save_csv()` writes rows to CSV

In [5]:
def print_stats(rows):
    """Print dataset summary including class distribution and corpus source breakdown."""
    print('\n--- Dataset statistics ---')
    print(f'Total pairs: {len(rows)}')

    label_counts  = Counter(r['interaction_type'] for r in rows)
    source_counts = Counter(r.get('corpus_source', 'Unknown') for r in rows)

    # ddi is now stored as bool — compare to bool not string
    ddi_true  = sum(1 for r in rows if r['ddi'] is True)
    ddi_false = sum(1 for r in rows if r['ddi'] is False)

    print(f'DDI = true:  {ddi_true}')
    print(f'DDI = false: {ddi_false}')

    # show breakdown by corpus source
    print('\nPairs by corpus source:')
    for src, count in source_counts.most_common():
        print(f'{src:<12} {count:6d}')

    print('\nInteraction type breakdown:')
    for label, count in label_counts.most_common():
        bar = chr(9608) * (count // 50)
        print(f'{label:<15} {count:5d}  {bar}')

def save_csv(rows, output_path):
    """Save parsed rows to CSV. Includes corpus_source and offset_valid columns."""
    if not rows:
        print('No data to save.')
        return

    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)

    fieldnames = [
        'sentence_id', 'sentence_text',
        'drug1_id', 'drug1_text', 'drug1_type',
        'drug2_id', 'drug2_text', 'drug2_type',
        'ddi', 'interaction_type',
        'corpus_source',   
        'offset_valid',   
        'source_file',
    ]

    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f'Saved {len(rows)} rows -> {output_path}')

## 5. Run the Parser

### 5.1 — Parse Training Data

Processes all XML files in the `Train/` folder and saves the result to `train.csv`.

In [6]:
parse_errors.clear()  # Reset error log before each run

print('=' * 60)
print('Parsing TRAINING data...')
print('=' * 60)

train_rows = parse_ddi_folder(TRAIN_DIR)
print_stats(train_rows)
save_csv(train_rows, TRAIN_OUTPUT)

Parsing TRAINING data...
Found 714 XML files in "../../data/raw/DDICorpus-2013/Train"
19-norandrostenedione_ddi.xml                      ->    0 pairs
Abarelix_ddi.xml                                   ->    1 pairs
Abatacept_ddi.xml                                  ->   81 pairs
Abciximab_ddi.xml                                  ->   35 pairs
Acamprosate_ddi.xml                                ->   31 pairs
Acarbose_ddi.xml                                   ->  115 pairs
Acebutolol_ddi.xml                                 ->   27 pairs
Acetazolamide_ddi.xml                              ->   16 pairs
Acetohydroxamic Acid_ddi.xml                       ->    1 pairs
Aciclovir_ddi.xml                                  ->    1 pairs
Acitretin_ddi.xml                                  ->   55 pairs
Adalimumab_ddi.xml                                 ->   14 pairs
Adapalene_ddi.xml                                  ->    6 pairs
Adefovir Dipivoxil_ddi.xml                         ->   36 pairs
Aden

### 5.2 — Parse Test Data

Processes all XML files in the `Test/` folder and saves the result to `test.csv`.

In [7]:
parse_errors.clear()  # Reset error log before each run

print('=' * 60)
print('Parsing TEST data...')
print('=' * 60)

test_rows = parse_ddi_folder(TEST_DIR)
print_stats(test_rows)
save_csv(test_rows, TEST_OUTPUT)

Parsing TEST data...
Found 303 XML files in "../../data/raw/DDICorpus-2013/Test"
Abacavir.xml                                       ->   23 pairs
Ezetimibe.xml                                      ->   52 pairs
Fluconazole.xml                                    ->  216 pairs
Mazindol.xml                                       ->  176 pairs
Mebendazole.xml                                    ->    3 pairs
Mecamylamine.xml                                   ->    6 pairs
Medroxyprogesterone.xml                            ->    3 pairs
Megestrol.xml                                      ->   13 pairs
Melatonin.xml                                      ->  136 pairs
Meloxicam.xml                                      ->   63 pairs
Melphalan.xml                                      ->    7 pairs
Mepenzolate.xml                                    ->  110 pairs
Meperidine.xml                                     ->   55 pairs
Mephenytoin.xml                                    ->   28 pairs
Mequitazi

## 6. Spot Check — Inspect Sample Rows

Always manually inspect a few rows after parsing to confirm the output looks correct before moving to Phase 2.

Things to verify:
- `sentence_text` contains real medical sentences
- `drug1_text` / `drug2_text` are actual drug names
- `interaction_type` is one of: `false`, `effect`, `mechanism`, `advise`, `int`
- `corpus_source` is either `DrugBank` or `MedLine`
- `offset_valid` is `True` for most rows

In [8]:
train_df = pd.read_csv(TRAIN_OUTPUT)

print(f'Training set shape: {train_df.shape}')
print(f'Columns: {list(train_df.columns)}')
print()
train_df.head(3)
print(train_df['ddi'].unique())
print(train_df['ddi'].value_counts())
print(len(train_df[train_df['ddi'] == True]))

Training set shape: (27792, 13)
Columns: ['sentence_id', 'sentence_text', 'drug1_id', 'drug1_text', 'drug1_type', 'drug2_id', 'drug2_text', 'drug2_type', 'ddi', 'interaction_type', 'corpus_source', 'offset_valid', 'source_file']

[False  True]
ddi
False    23771
True      4021
Name: count, dtype: int64
4021


In [9]:
print('--- Positive interaction example ---')

pos_df = train_df[train_df['ddi']]

if not pos_df.empty:
    pos = pos_df.iloc[0]

    print(f'Sentence : {pos["sentence_text"]}')
    print(f'Drug 1 : {pos["drug1_text"]}')
    print(f'Drug 2 : {pos["drug2_text"]}')
    print(f'Type : {pos["interaction_type"]}')
    print(f'Corpus source : {pos["corpus_source"]}')
    print(f'Offset valid : {pos["offset_valid"]}')
else:
    print("No positive samples found ")


print('\n--- Negative (no interaction) example ---')

neg_df = train_df[~train_df['ddi']]

if not neg_df.empty:
    neg = neg_df.iloc[0]

    print(f'Sentence : {neg["sentence_text"]}')
    print(f'Drug 1 : {neg["drug1_text"]}')
    print(f'Drug 2 : {neg["drug2_text"]}')
    print(f'Type : {neg["interaction_type"]}')
    print(f'Corpus source : {neg["corpus_source"]}')
else:
    print("No negative samples found ")

--- Positive interaction example ---
Sentence : Concurrent administration of a TNF antagonist with ORENCIA has been associated with an increased risk of serious infections and no significant additional efficacy over use of the TNF antagonists alone.
Drug 1 : TNF antagonist
Drug 2 : ORENCIA
Type : effect
Corpus source : DrugBank
Offset valid : True

--- Negative (no interaction) example ---
Sentence : Laboratory Tests Response to Plenaxis should be monitored by measuring serum total testosterone concentrations just prior to administration on Day 29 and every 8 weeks thereafter.
Drug 1 : Plenaxis
Drug 2 : testosterone
Type : false
Corpus source : DrugBank


### Corpus Source Breakdown (Optimisation B)

DrugBank and MedLine text have different characteristics:
- **DrugBank** text is structured and precise — written by pharmacologists
- **MedLine** text is free-form PubMed abstracts — more varied language

This breakdown lets you later analyse whether your model performs differently on each source.

In [10]:
print('--- Class distribution by corpus source ---\n')
pivot = train_df.groupby(['corpus_source', 'interaction_type']).size().unstack(fill_value=0)
print(pivot.to_string())

print('\n--- Offset validity summary ---')
invalid = train_df[train_df['offset_valid'] == False]
print(f'Total rows : {len(train_df)}')
print(f'Offset valid : {(train_df["offset_valid"] == True).sum()}')
print(f'Offset invalid : {len(invalid)}')
if len(invalid) > 0:
    print(f'Invalid % of total : {len(invalid)/len(train_df)*100:.2f}%')
    print('First 3 invalid rows:')
    print(invalid[['sentence_id','drug1_text','drug2_text','interaction_type']].head(3).to_string())

--- Class distribution by corpus source ---

interaction_type  advise  effect  false  int  mechanism
corpus_source                                          
DrugBank             818    1535  22216  179       1257
MedLine                8     152   1555   10         62

--- Offset validity summary ---
Total rows : 27792
Offset valid : 27663
Offset invalid : 129
Invalid % of total : 0.46%
First 3 invalid rows:
               sentence_id                             drug1_text                         drug2_text interaction_type
116   DDI-DrugBank.d532.s4       diagnostic monoclonal antibodies  therapeutic monoclonal antibodies            false
1169  DDI-DrugBank.d356.s4  non-steroidal anti-inflammatory agent                     loop diuretics           effect
1170  DDI-DrugBank.d356.s4  non-steroidal anti-inflammatory agent        potassium-sparing diuretics           effect


## 7. Automated Quality Checks

These checks validate the output before passing to Phase 2. All checks must pass before proceeding.

In [11]:
print('--- Quality checks ---\n')
passed = True

# Check 1: No missing values in critical columns
critical_cols = ['sentence_text', 'drug1_text', 'drug2_text', 'interaction_type']
missing = train_df[critical_cols].isnull().sum()
print('1. Missing values in critical columns:')
print(missing.to_string())
if missing.sum() > 0:
    print('FAIL')
    passed = False
else:
    print('PASS')

# Check 2: Only valid labels present
valid_labels = {'false', 'effect', 'mechanism', 'advise', 'int'}
actual_labels = set(train_df['interaction_type'].unique())
unexpected = actual_labels - valid_labels
print(f'\n2. Unexpected labels: {unexpected if unexpected else "none"}')
print('PASS' if not unexpected else 'FAIL')
if unexpected:
    passed = False

# Check 3: No empty sentences
empty_sents = (train_df['sentence_text'].str.strip() == '').sum()
print(f'\n3. Empty sentences: {empty_sents}')
print('PASS' if empty_sents == 0 else 'FAIL')
if empty_sents > 0:
    passed = False

# No empty drug names
empty_drugs = ((train_df['drug1_text'].str.strip() == '') |
               (train_df['drug2_text'].str.strip() == '')).sum()
print(f'\n4. Rows with empty drug names: {empty_drugs}')
print('PASS' if empty_drugs == 0 else 'FAIL')
if empty_drugs > 0:
    passed = False

# corpus_source column populated correctly
unknown_source = (train_df['corpus_source'] == 'Unknown').sum()
print(f'\n5. Rows with unknown corpus source: {unknown_source}')
print('PASS' if unknown_source == 0 else 'WARN — some files could not be classified')

# parse errors
print(f'\n6. Files that failed to parse: {len(parse_errors)}')
print('PASS' if len(parse_errors) == 0 else f'WARN — {len(parse_errors)} file(s) skipped')

print('\n' + ('=' * 45))
print('All checks passed. Ready for Phase 2!' if passed else 'Issues found — review output above.')
print('=' * 45)

--- Quality checks ---

1. Missing values in critical columns:
sentence_text       0
drug1_text          0
drug2_text          0
interaction_type    0
PASS

2. Unexpected labels: none
PASS

3. Empty sentences: 0
PASS

4. Rows with empty drug names: 0
PASS

5. Rows with unknown corpus source: 0
PASS

6. Files that failed to parse: 0
PASS

All checks passed. Ready for Phase 2!


## 8. Summary

| Item | Detail |
|---|---|
| **Source** | SemEval-2013 DDI Corpus (CC BY-NC licence) |
| **Output files** | `train.csv`, `test.csv` |
| **Label classes** | `false`, `effect`, `mechanism`, `advise`, `int` |
| **New columns** | `corpus_source`, `offset_valid` |
| **Class imbalance** | ~85% `false` — handled in Phase 4 with `class_weight='balanced'` |
